# EMBER2024 SHAP Chunk Generator

This notebook mirrors `shap_chunk_codex.ipynb`, but targets vectorized EMBER2024 data.

It loads one EMBER2024 platform split (`Win64` or `Win32`), selects a deterministic fraction of rows, computes LightGBM SHAP values for one row chunk at a time, saves each chunk as a pickle, and optionally merges all completed chunks into one full SHAP pickle.

Run one `SHAP_CHUNK_INDEX` at a time. After all chunks exist, set `COMPUTE_CURRENT_CHUNK = False` and `MERGE_SHAP_CHUNKS = True`, then run the merge cell.

Notes:

- Current implementation supports LightGBM joblib artifacts with a fitted `LGBMClassifier` or native `Booster`, plus native LightGBM `.model` / `.txt` model files.
- The subset indices are saved as `.npy` so later attack notebooks can use the exact same rows.
- Chunking rows does not change LightGBM per-row SHAP values when chunks are merged back in the selected-row order.


In [ ]:

from pathlib import Path
import hashlib
import json
import os
import random
import sys
import time


def find_repo_root(start=None):
    start = Path(start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if (candidate / 'backdoor_attack.py').exists() and (candidate / 'mw_backdoor').is_dir():
            return candidate
    raise RuntimeError('Could not locate repository root containing backdoor_attack.py and mw_backdoor/')


REPO_ROOT = find_repo_root()
os.chdir(REPO_ROOT)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

# Keep Matplotlib/EMBER2024 imports from attempting to write into the home directory.
os.environ.setdefault('MPLCONFIGDIR', str(REPO_ROOT / 'build' / 'matplotlib'))
os.makedirs(os.environ['MPLCONFIGDIR'], exist_ok=True)

EMBER2024_ROOT_CANDIDATES = [
    REPO_ROOT.parent / 'ember2024' / 'EMBER2024',
    REPO_ROOT.parent / 'ember2024' / 'ember2024',
    Path('../ember2024/EMBER2024').resolve(),
    Path('../ember2024/ember2024').resolve(),
]
EMBER2024_ROOT = next((p for p in EMBER2024_ROOT_CANDIDATES if p.exists()), None)
if EMBER2024_ROOT is None:
    raise FileNotFoundError('Could not locate EMBER2024 root. Tried: {}'.format(EMBER2024_ROOT_CANDIDATES))
EMBER2024_SRC_DIR = EMBER2024_ROOT / 'src'
EMBER2024_EXAMPLES_DIR = EMBER2024_ROOT / 'examples'
if str(EMBER2024_SRC_DIR) not in sys.path:
    sys.path.insert(0, str(EMBER2024_SRC_DIR))
if str(EMBER2024_EXAMPLES_DIR) not in sys.path:
    sys.path.insert(0, str(EMBER2024_EXAMPLES_DIR))

import lightgbm as lgb
import numpy as np
import pandas as pd
from joblib import load as joblib_load
from sklearn.model_selection import train_test_split

import thrember

SEED = 42
EMBER2024_FEATURE_DIM = thrember.PEFeatureExtractor().dim

print('Repository root:', REPO_ROOT)
print('EMBER2024 root:', EMBER2024_ROOT)
print('EMBER2024 feature count:', EMBER2024_FEATURE_DIM)


Repository root: /Users/falcon/Machine Learning/Severi Data Poisoning Attack
EMBER2024 root: /Users/falcon/Machine Learning/ember2024/EMBER2024
EMBER2024 feature count: 2568


In [ ]:

# Dataset/model settings.
EMBER2024_PLATFORM = 'Win64'  # 'Win64' or 'Win32'
SHAP_DATA_SPLIT = 'train'     # Usually 'train' for attack feature selection.
MODEL_ALGORITHM = 'lightgbm'
MODEL_PATH_BY_PLATFORM = {
    'Win64': EMBER2024_EXAMPLES_DIR / 'lgbm_win64_retrained.joblib',
    # Add your Win32 model path here when available. Both joblib and native LightGBM files work, for example:
    # 'Win32': EMBER2024_EXAMPLES_DIR / 'lgbm_win32_retrained.joblib',
    # 'Win32': EMBER2024_EXAMPLES_DIR / 'xxx.model',
    'Win32': None,
}
MODEL_PATH = MODEL_PATH_BY_PLATFORM.get(EMBER2024_PLATFORM)

# Fraction of labeled rows to explain. Examples: 0.20, 0.30, 0.50, 1.0.
SHAP_PERCENT = 0.20

# Subset mode:
# - 'head': first N labeled rows, closest to the original shap_chunk_codex.ipynb behavior.
# - 'stratified_random': deterministic stratified subset using SEED, useful for paper experiments.
SUBSET_MODE = 'stratified_random'

# For 5% chunks use 20 chunks; for 2% chunks use 50 chunks; adjust for your storage/runtime.
SHAP_NUM_CHUNKS = 20
SHAP_CHUNK_INDEX = 19

# Set False when you only want to merge already-created chunks.
COMPUTE_CURRENT_CHUNK = True

# Set True only after all chunk pickle files exist.
MERGE_SHAP_CHUNKS = True

# Set True if you intentionally want to recompute an existing chunk file.
OVERWRITE_CHUNK = False

# Output location.
SHAP_CACHE_DIR = REPO_ROOT / 'ember2024_shap_cache' / EMBER2024_PLATFORM
SHAP_CACHE_DIR.mkdir(parents=True, exist_ok=True)

if MODEL_PATH is None:
    raise FileNotFoundError('MODEL_PATH is not configured for platform {}'.format(EMBER2024_PLATFORM))
MODEL_PATH = Path(MODEL_PATH)
if not MODEL_PATH.exists():
    raise FileNotFoundError('Missing model file: {}'.format(MODEL_PATH))
if not 0 < float(SHAP_PERCENT) <= 1.0:
    raise ValueError('SHAP_PERCENT must be in (0, 1].')
if SUBSET_MODE not in {'head', 'stratified_random'}:
    raise ValueError("SUBSET_MODE must be 'head' or 'stratified_random'.")

print('EMBER2024_PLATFORM:', EMBER2024_PLATFORM)
print('SHAP_DATA_SPLIT:', SHAP_DATA_SPLIT)
print('MODEL_ALGORITHM:', MODEL_ALGORITHM)
print('MODEL_PATH:', MODEL_PATH)
print('SHAP_PERCENT:', SHAP_PERCENT)
print('SUBSET_MODE:', SUBSET_MODE)
print('SHAP_NUM_CHUNKS:', SHAP_NUM_CHUNKS)
print('SHAP_CHUNK_INDEX:', SHAP_CHUNK_INDEX)
print('SHAP_CACHE_DIR:', SHAP_CACHE_DIR)


EMBER2024_PLATFORM: Win32
SHAP_DATA_SPLIT: train
MODEL_ALGORITHM: lightgbm
MODEL_PATH: /Users/falcon/Machine Learning/ember2024/EMBER2024/Model/EMBER2024_Win32.model
SHAP_PERCENT: 0.2
SUBSET_MODE: stratified_random
SHAP_NUM_CHUNKS: 20
SHAP_CHUNK_INDEX: 0
SHAP_CACHE_DIR: /Users/falcon/Machine Learning/Severi Data Poisoning Attack/ember2024_shap_cache/Win32


In [3]:

random.seed(SEED)
np.random.seed(SEED)


def file_sha256(path, block_size=1024 * 1024):
    digest = hashlib.sha256()
    with open(path, 'rb') as f:
        for block in iter(lambda: f.read(block_size), b''):
            digest.update(block)
    return digest.hexdigest()


def fraction_tag(value):
    return ('{:.4f}'.format(float(value))).rstrip('0').rstrip('.').replace('.', 'p')


def load_ember2024_split(platform, split):
    data_dir = EMBER2024_ROOT / 'data' / platform
    x_path = data_dir / 'X_{}.dat'.format(split)
    y_path = data_dir / 'y_{}.dat'.format(split)
    if not x_path.exists() or not y_path.exists():
        raise FileNotFoundError('Missing EMBER2024 files for {} {} in {}'.format(platform, split, data_dir))
    x_memmap = np.memmap(x_path, dtype=np.float32, mode='r').reshape(-1, EMBER2024_FEATURE_DIM)
    y_memmap = np.memmap(y_path, dtype=np.int32, mode='r')
    if y_memmap.shape[0] != x_memmap.shape[0]:
        raise ValueError('X/y row mismatch: {} vs {}'.format(x_memmap.shape[0], y_memmap.shape[0]))
    return x_memmap, np.asarray(y_memmap, dtype=np.int32)


def select_labeled_indices(y, percent, mode, seed):
    labeled_indices = np.flatnonzero(y != -1)
    labeled_y = y[labeled_indices]
    selected_count = int(round(labeled_indices.shape[0] * float(percent)))
    selected_count = max(np.unique(labeled_y).shape[0], selected_count)
    selected_count = min(selected_count, labeled_indices.shape[0])
    if selected_count == labeled_indices.shape[0]:
        return labeled_indices
    if mode == 'head':
        return labeled_indices[:selected_count]
    selected_indices, _, _, _ = train_test_split(
        labeled_indices,
        labeled_y,
        train_size=selected_count,
        random_state=seed,
        stratify=labeled_y,
    )
    return np.sort(selected_indices)


def load_lightgbm_model(model_path):
    model_path = Path(model_path)

    if model_path.suffix.lower() in {'.model', '.txt'}:
        booster = lgb.Booster(model_file=str(model_path))
        metadata = {
            'algorithm': 'lightgbm',
            'artifact_type': 'native_lightgbm_model',
            'model_path': str(model_path),
        }
    else:
        artifact = joblib_load(model_path)
        if isinstance(artifact, dict) and 'model' in artifact:
            model = artifact['model']
            algorithm = artifact.get('algorithm', MODEL_ALGORITHM)
            metadata = {k: v for k, v in artifact.items() if k != 'model'}
        else:
            model = artifact
            algorithm = MODEL_ALGORITHM
            metadata = {'artifact_type': model.__class__.__name__}
        if algorithm != 'lightgbm':
            raise NotImplementedError('This notebook currently supports LightGBM SHAP only, got {}'.format(algorithm))

        booster = getattr(model, 'booster_', None)
        if booster is None and model.__class__.__module__.startswith('lightgbm') and model.__class__.__name__ == 'Booster':
            booster = model
        if booster is None:
            raise ValueError(
                'Expected a fitted LGBMClassifier with booster_, a joblib-saved LightGBM Booster, '
                'or a native LightGBM .model/.txt file.'
            )

    if booster.num_feature() != EMBER2024_FEATURE_DIM:
        raise ValueError('Model expects {} features, but EMBER2024 has {}'.format(
            booster.num_feature(), EMBER2024_FEATURE_DIM
        ))
    return booster, metadata


x_all, y_all = load_ember2024_split(EMBER2024_PLATFORM, SHAP_DATA_SPLIT)
selected_indices = select_labeled_indices(y_all, SHAP_PERCENT, SUBSET_MODE, SEED)
model_hash = file_sha256(MODEL_PATH)[:12]
base_cache_key = 'ember2024_{}_{}_{}_{}_{}_seed{}_model{}'.format(
    EMBER2024_PLATFORM.lower(),
    MODEL_ALGORITHM,
    SHAP_DATA_SPLIT,
    fraction_tag(SHAP_PERCENT),
    SUBSET_MODE,
    SEED,
    model_hash,
)
merged_shap_path = SHAP_CACHE_DIR / 'shap_values_{}.pkl'.format(base_cache_key)
subset_indices_path = SHAP_CACHE_DIR / 'indices_{}.npy'.format(base_cache_key)
subset_metadata_path = SHAP_CACHE_DIR / 'indices_{}.json'.format(base_cache_key)

np.save(subset_indices_path, selected_indices)
label_values, label_counts = np.unique(y_all[selected_indices], return_counts=True)
subset_metadata = {
    'base_cache_key': base_cache_key,
    'platform': EMBER2024_PLATFORM,
    'split': SHAP_DATA_SPLIT,
    'feature_dim': int(EMBER2024_FEATURE_DIM),
    'total_rows': int(x_all.shape[0]),
    'labeled_rows': int(np.flatnonzero(y_all != -1).shape[0]),
    'selected_rows': int(selected_indices.shape[0]),
    'shap_percent': float(SHAP_PERCENT),
    'subset_mode': SUBSET_MODE,
    'seed': int(SEED),
    'label_counts': {str(int(k)): int(v) for k, v in zip(label_values, label_counts)},
    'model_path': str(MODEL_PATH),
    'model_sha256_12': model_hash,
}
subset_metadata_path.write_text(json.dumps(subset_metadata, indent=2, sort_keys=True) + '\n')

original_model, model_metadata = load_lightgbm_model(MODEL_PATH)

print('Full source X:', x_all.shape)
print('Full source y:', y_all.shape)
print('Selected rows:', selected_indices.shape[0])
print('Selected label counts:', subset_metadata['label_counts'])
print('Subset indices path:', subset_indices_path)
print('Merged SHAP path:', merged_shap_path)
print('Model metadata:', model_metadata)


IndexError: pop from empty list

In [ ]:

def chunk_bounds(n_rows, num_chunks, chunk_index):
    if not 0 <= chunk_index < num_chunks:
        raise ValueError('chunk_index must be in [0, num_chunks)')
    start = n_rows * chunk_index // num_chunks
    end = n_rows * (chunk_index + 1) // num_chunks
    return start, end


def lightgbm_shap_dataframe(booster, x_chunk):
    contribs = booster.predict(x_chunk, pred_contrib=True)
    shap_values = np.asarray(contribs, dtype=np.float32)
    if shap_values.ndim != 2:
        raise ValueError('Expected 2D LightGBM pred_contrib output, got {}'.format(shap_values.shape))
    if shap_values.shape[1] == x_chunk.shape[1] + 1:
        shap_values = shap_values[:, :-1]
    if shap_values.shape != x_chunk.shape:
        raise ValueError('SHAP shape {} does not match chunk shape {}'.format(shap_values.shape, x_chunk.shape))
    return pd.DataFrame(shap_values)


chunk_start, chunk_end = chunk_bounds(selected_indices.shape[0], SHAP_NUM_CHUNKS, SHAP_CHUNK_INDEX)
chunk_selected_indices = selected_indices[chunk_start:chunk_end]
chunk_path = SHAP_CACHE_DIR / 'shap_values_{}_chunk_{:03d}_of_{:03d}.pkl'.format(
    base_cache_key,
    SHAP_CHUNK_INDEX,
    SHAP_NUM_CHUNKS,
)
metadata_path = chunk_path.with_suffix('.json')

print('Chunk selected-row positions: [{}:{})'.format(chunk_start, chunk_end))
print('Chunk original row index count:', chunk_selected_indices.shape[0])
print('Chunk path:', chunk_path)

if COMPUTE_CURRENT_CHUNK:
    if chunk_path.exists() and not OVERWRITE_CHUNK:
        print('Chunk already exists and OVERWRITE_CHUNK is False. Skipping compute.')
    else:
        print('Loading chunk feature rows...')
        start_time = time.time()
        x_chunk = np.asarray(x_all[chunk_selected_indices], dtype=np.float32)
        print('Loading chunk rows took {:.2f} seconds'.format(time.time() - start_time))
        print('Computing SHAP for chunk shape:', x_chunk.shape)
        start_time = time.time()
        shap_chunk_df = lightgbm_shap_dataframe(original_model, x_chunk)
        print('Computing chunk SHAP took {:.2f} seconds'.format(time.time() - start_time))
        assert shap_chunk_df.shape == x_chunk.shape, (
            'Chunk SHAP shape {} does not match chunk data shape {}'.format(shap_chunk_df.shape, x_chunk.shape)
        )
        # Index is relative to the selected subset, not the original EMBER2024 row number.
        shap_chunk_df.index = range(chunk_start, chunk_end)
        start_time = time.time()
        shap_chunk_df.to_pickle(chunk_path)
        metadata = {
            'base_cache_key': base_cache_key,
            'chunk_index': int(SHAP_CHUNK_INDEX),
            'num_chunks': int(SHAP_NUM_CHUNKS),
            'selected_row_start': int(chunk_start),
            'selected_row_end': int(chunk_end),
            'original_row_min': int(chunk_selected_indices.min()) if chunk_selected_indices.size else None,
            'original_row_max': int(chunk_selected_indices.max()) if chunk_selected_indices.size else None,
            'shape': list(shap_chunk_df.shape),
            'platform': EMBER2024_PLATFORM,
            'model_algorithm': MODEL_ALGORITHM,
            'shap_data_split': SHAP_DATA_SPLIT,
            'shap_percent': float(SHAP_PERCENT),
            'subset_mode': SUBSET_MODE,
            'seed': int(SEED),
            'feature_dim': int(EMBER2024_FEATURE_DIM),
            'model_path': str(MODEL_PATH),
            'model_sha256_12': model_hash,
            'subset_indices_path': str(subset_indices_path),
        }
        metadata_path.write_text(json.dumps(metadata, indent=2, sort_keys=True) + '\n')
        print('Saving chunk pickle took {:.2f} seconds'.format(time.time() - start_time))
        print('Saved:', chunk_path)
        print('Saved metadata:', metadata_path)
else:
    print('COMPUTE_CURRENT_CHUNK is False. No chunk was computed.')


Chunk selected-row positions: [197600:208000)
Chunk original row index count: 10400
Chunk path: /Users/falcon/Machine Learning/Severi Data Poisoning Attack/ember2024_shap_cache/Win64/shap_values_ember2024_win64_lightgbm_train_0p2_stratified_random_seed42_model832cb33e59af_chunk_019_of_020.pkl
Chunk already exists and OVERWRITE_CHUNK is False. Skipping compute.


In [ ]:

chunk_paths = [
    SHAP_CACHE_DIR / 'shap_values_{}_chunk_{:03d}_of_{:03d}.pkl'.format(
        base_cache_key,
        chunk_index,
        SHAP_NUM_CHUNKS,
    )
    for chunk_index in range(SHAP_NUM_CHUNKS)
]
missing_chunks = [path for path in chunk_paths if not path.exists()]

print('Expected chunks:', len(chunk_paths))
print('Missing chunks:', len(missing_chunks))
if missing_chunks:
    print('First missing chunk:', missing_chunks[0])

if MERGE_SHAP_CHUNKS:
    if missing_chunks:
        raise FileNotFoundError('Cannot merge because {} chunk(s) are missing.'.format(len(missing_chunks)))

    start_time = time.time()
    print('Loading and merging chunk pickles...')
    shap_values_df = pd.concat((pd.read_pickle(path) for path in chunk_paths), axis=0)
    shap_values_df = shap_values_df.sort_index().reset_index(drop=True)
    expected_shape = (selected_indices.shape[0], EMBER2024_FEATURE_DIM)
    assert shap_values_df.shape == expected_shape, (
        'Merged SHAP shape {} does not match expected shape {}'.format(shap_values_df.shape, expected_shape)
    )
    shap_values_df.to_pickle(merged_shap_path)
    merge_metadata = dict(subset_metadata)
    merge_metadata.update({
        'num_chunks': int(SHAP_NUM_CHUNKS),
        'merged_shape': list(shap_values_df.shape),
        'merged_path': str(merged_shap_path),
    })
    merged_shap_path.with_suffix('.json').write_text(json.dumps(merge_metadata, indent=2, sort_keys=True) + '\n')
    print('Merged SHAP pickle took {:.2f} seconds'.format(time.time() - start_time))
    print('Merged SHAP shape:', shap_values_df.shape)
    print('Saved merged pickle:', merged_shap_path)
else:
    print('MERGE_SHAP_CHUNKS is False. No merge was performed.')


Expected chunks: 20
Missing chunks: 0
Loading and merging chunk pickles...
Merged SHAP pickle took 5.42 seconds
Merged SHAP shape: (208000, 2568)
Saved merged pickle: /Users/falcon/Machine Learning/Severi Data Poisoning Attack/ember2024_shap_cache/Win64/shap_values_ember2024_win64_lightgbm_train_0p2_stratified_random_seed42_model832cb33e59af.pkl
